# Amazon ML Challenge 2026 — Business Entity Resolution
## Main Experiment + Integration Notebook

This notebook is the team's single research/integration notebook.

**Important:** keep the official `student_resource/` folder untouched. This notebook reads it;
it does not modify the challenge TSVs or the official validator.

### Project flow

`Source 1 → blocking → candidate pairs → pairwise features → ML matcher → threshold → all accepted matches`

The challenge is **multi-match**: a Source-1 entity may have zero, one, or many correct Source-2/3 matches.

We therefore evaluate with the official challenge's **macro F0.5 mindset**, while also tracking candidate recall,
pair-level diagnostics, false positives, and false negatives.

In [ ]:
# =========================
# 0. Environment & paths
# =========================
from pathlib import Path
import sys
import os

# Colab convenience: uncomment if needed.
# from google.colab import drive
# drive.mount("/content/drive")

# CHANGE THIS ONE PATH.
# Example:
# STUDENT_RESOURCE = Path("/content/drive/MyDrive/Amazon ML Challenge 2026/student_resource")
STUDENT_RESOURCE = Path("/content/drive/MyDrive/Amazon ML Challenge 2026/student_resource")

PROJECT_ROOT = STUDENT_RESOURCE.parent
OUTPUT_ROOT = PROJECT_ROOT / "amazon_ml_project" / "outputs"

sys.path.insert(0, str(PROJECT_ROOT / "amazon_ml_project"))

print("Student resource:", STUDENT_RESOURCE)
print("Exists:", STUDENT_RESOURCE.exists())

In [ ]:
# =========================
# 1. Imports
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import REQUIRED_COLUMNS, GROUND_TRUTH_COLUMNS
from src.io_utils import iter_tsv, read_tsv, validate_columns
from src.evaluation import parse_ground_truth, macro_f05, candidate_recall, threshold_sweep

TRAIN_DIR = STUDENT_RESOURCE / "dataset" / "train"
TEST_DIR = STUDENT_RESOURCE / "dataset" / "test"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
GROUND_TRUTH = TRAIN_DIR / "train_ground_truth.tsv"

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

for p in [TRAIN_S1, TRAIN_S2, TRAIN_S3, GROUND_TRUTH, TEST_S1, TEST_S2, TEST_S3]:
    print(("OK   " if p.exists() else "MISS "), p)

## 2. Data loading strategy

The full challenge data is large. Do **not** blindly load every TSV into pandas at once.

We will:
- use full loading only when memory allows and the table is genuinely needed;
- otherwise stream through TSVs in chunks;
- select only required columns;
- build compact indexes for blocking rather than keeping unnecessary duplicate copies of text.

In [ ]:
# =========================
# 2.1 Load Source 1 + Ground Truth for the first audit
# =========================
s1 = read_tsv(TRAIN_S1)
gt = read_tsv(GROUND_TRUTH)

validate_columns(s1.columns, REQUIRED_COLUMNS, context="train_source1")
validate_columns(gt.columns, GROUND_TRUTH_COLUMNS, context="train_ground_truth")

print("Source 1 shape:", s1.shape)
print("Ground truth shape:", gt.shape)
print("\nSource 1 columns:", list(s1.columns))
print("Ground truth columns:", list(gt.columns))

In [ ]:
# =========================
# 3. Source 1 integrity audit
# =========================
audit = {
    "rows": len(s1),
    "unique_entity_id": s1["entity_id"].nunique(),
    "duplicate_entity_id_rows": int(s1["entity_id"].duplicated().sum()),
    "missing_business_name": int(s1["business_name"].isna().sum()),
    "missing_business_address": int(s1["business_address"].isna().sum()),
    "missing_country": int(s1["country"].isna().sum()),
}
pd.Series(audit, name="value")

In [ ]:
# =========================
# 4. Ground truth audit
# =========================
gt_map = parse_ground_truth(gt)
match_counts = pd.Series({k: len(v) for k, v in gt_map.items()}, name="match_count")

print("S1 entities in ground truth:", len(gt_map))
print("Unmatched:", int((match_counts == 0).sum()))
print("Singletons:", int((match_counts == 1).sum()))
print("Maximum matches for one S1:", int(match_counts.max()))
print("Mean matches:", match_counts.mean())
print("Median matches:", match_counts.median())

display(match_counts.value_counts().sort_index().to_frame("s1_entities"))

In [ ]:
# =========================
# 5. Ground truth integrity checks
# =========================
bad_s1 = set(gt["source1_entity_id"]) - set(s1["entity_id"])
missing_gt = set(s1["entity_id"]) - set(gt["source1_entity_id"])
duplicate_gt_ids = int(gt["source1_entity_id"].duplicated().sum())

match_id_examples = (
    gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
)
match_id_examples = match_id_examples[match_id_examples != ""]

print("Ground truth rows with duplicate S1 IDs:", duplicate_gt_ids)
print("Ground truth S1 IDs not in Source 1:", len(bad_s1))
print("Source 1 IDs missing from ground truth:", len(missing_gt))
print("Example target IDs:", match_id_examples.head(10).tolist())

## 6. Validation design

Before training the final matcher, reserve a reproducible validation population from the training entities.

The validation set's ground truth remains available **only for evaluation**. The matching labels must not be used
to construct the features/model for those validation rows.

We will later run the entire candidate-generation + matching pipeline on validation rows and report:

- candidate recall
- macro F0.5
- global precision/recall as diagnostics
- false-positive count
- false-negative count
- behavior on unmatched S1 entities
- behavior as the number of true matches increases

The exact split strategy should be finalized after Source 2/3 profiling.

In [ ]:
# =========================
# 7. Reproducible split scaffold
# =========================
RANDOM_SEED = 42
VALIDATION_FRAC = 0.15

rng = np.random.default_rng(RANDOM_SEED)
all_ids = s1["entity_id"].astype(str).to_numpy()

validation_size = int(len(all_ids) * VALIDATION_FRAC)
validation_ids = set(rng.choice(all_ids, size=validation_size, replace=False))

print("Validation S1 entities:", len(validation_ids))
print("Development S1 entities:", len(all_ids) - len(validation_ids))

## 8. Source 2 / Source 3 profiling — next stage

Once the complete `train_source2.tsv` and `train_source3.tsv` are available in the shared environment,
run this section.

We will specifically measure:

1. row counts and memory footprint
2. missingness and duplicate IDs
3. country distribution
4. name/address characteristics
5. exact-normalized name overlap with S1
6. exact-normalized address overlap
7. how many true matches survive each blocking rule
8. candidate volume produced by each blocking rule

**Do not pick a final blocking strategy before these measurements.**

In [ ]:
# =========================
# 8.1 Minimal source profiling helper
# =========================
from src.io_utils import count_tsv_rows

def quick_tsv_profile(path, sample_rows=200_000):
    print("\nFILE:", path)
    print("Rows:", count_tsv_rows(path))
    sample = read_tsv(path, nrows=sample_rows)
    print("Sample shape:", sample.shape)
    print("Columns:", list(sample.columns))
    print("Missingness (sample):")
    display(sample.isna().sum().to_frame("missing"))
    if "country" in sample:
        print("Countries (sample):")
        display(sample["country"].value_counts(dropna=False).head(20).to_frame("rows"))
    return sample

# Uncomment after confirming the files are available:
# s2_sample = quick_tsv_profile(TRAIN_S2)
# s3_sample = quick_tsv_profile(TRAIN_S3)

## 9. Normalization experiments

Use conservative normalization first. Keep both raw and normalized values available.

Potential layers:
- Unicode/case normalization
- punctuation/whitespace normalization
- optional legal-suffix stripping
- conservative address abbreviation normalization

We should measure whether each layer increases true-match similarity without creating too many false positives.

In [ ]:
# =========================
# 9.1 Example normalization inspection
# =========================
from src.normalization import normalize_name, normalize_address

demo = s1[["entity_id", "business_name", "business_address"]].head(20).copy()
demo["name_norm"] = demo["business_name"].map(normalize_name)
demo["name_nosuffix"] = demo["business_name"].map(
    lambda x: normalize_name(x, strip_legal_suffixes=True)
)
demo["address_norm"] = demo["business_address"].map(normalize_address)

display(demo)

## 10. Blocking experiment framework

A blocking rule is successful only if it keeps true matches **and** reduces the search space.

For every candidate strategy record:

`strategy → candidate count → candidate recall → average candidates per S1`

The final `candidate_pairs.tsv` must be the exact candidate set actually sent into the final matcher.

## 11. Pairwise feature engineering

For each `(S1, candidate S2/S3)` pair we can compute:
- name similarity
- name similarity after legal-suffix normalization
- token overlap
- address similarity
- address token overlap
- country compatibility
- source indicator
- simple length diagnostics

Feature selection will be based on validation results, not intuition alone.

## 12. Model + threshold experiments

Train the pairwise matcher only on candidate pairs.

For every candidate pair, the label is:

`1 if candidate_entity_id is in that S1's ground-truth set else 0`

The model produces a probability/score.

Thresholds must be evaluated against the **same held-out validation set** and compared using macro F0.5.
Do not assume 0.50 is optimal.

## 13. Final inference

After validation is frozen:

`test_source1 → final blocking → final candidate set → feature generation → trained matcher → selected threshold → all accepted matches`

No external lookup or augmentation is part of the pipeline.

Then generate:
- `output/matching_results.tsv`
- `output/candidate_pairs.tsv`

Finally run the organizer-provided validator from the official `student_resource/utils/validate_submission.py`.

In [ ]:
# =========================
# 14. Submission schema reminder
# =========================
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]

print("Required matching headers:", MATCHING_HEADER)
print("Required candidate headers:", CANDIDATE_HEADER)

## 15. Experiment log

Record every materially different experiment.

Suggested columns:
- experiment_id
- blocking_version
- features_version
- model
- train_rows / candidate_pairs
- candidate_recall
- threshold
- macro_f05
- global_precision
- global_recall
- notes

In [ ]:
# =========================
# 15.1 Empty experiment log
# =========================
experiment_log = pd.DataFrame(columns=[
    "experiment_id",
    "blocking_version",
    "features_version",
    "model",
    "candidate_pairs",
    "candidate_recall",
    "threshold",
    "macro_f05",
    "global_precision",
    "global_recall",
    "notes",
])
display(experiment_log)